In [ ]:
# Imports.
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
from matplotlib import font_manager


In [ ]:
# Input, output, and method selection.
data_path = "/home/liyuanzhe/lyz32/3DCell-master/metrics_with_descriptors_WF_final.csv"
save_path = "./correlation_output/WF_metric_specific_top5/"
os.makedirs(save_path, exist_ok=True)
df = pd.read_csv(data_path)
AP_COLUMN = "AP"  # Plotted AP uses the rule-based score.
if AP_COLUMN not in df.columns:
    raise ValueError(f"Missing AP column: {AP_COLUMN}")
print(f"AP source: {AP_COLUMN}")
mt_list = ["Cellpose-SAM", "Cellpose-2D", "Cellpose", "CellSAM", "Stardist", "CellSeg3D", "uSAM"]
df = df[df["Method"].isin(mt_list)].copy()
print("Input dataframe shape:", df.shape)


In [ ]:
# Font, metric, descriptor, and figure settings.
FONT_SIZE = 12
ARIAL_FONT_PATH = None


def configure_font():
    if ARIAL_FONT_PATH is not None and os.path.isfile(ARIAL_FONT_PATH):
        font_manager.fontManager.addfont(ARIAL_FONT_PATH)
        font_property = font_manager.FontProperties(fname=ARIAL_FONT_PATH)
        font_family = font_property.get_name()
        print("Using registered font:", font_family)
        return font_family
    for candidate_font in ["Arial", "Liberation Sans", "DejaVu Sans"]:
        try:
            font_manager.findfont(candidate_font, fallback_to_default=False)
            print("Using font:", candidate_font)
            return candidate_font
        except ValueError:
            continue
    return "sans-serif"


FONT_FAMILY = configure_font()
plt.rcParams.update(
    {
        "font.family": FONT_FAMILY,
        "font.size": FONT_SIZE,
        "axes.titlesize": FONT_SIZE,
        "axes.labelsize": FONT_SIZE,
        "xtick.labelsize": FONT_SIZE,
        "ytick.labelsize": FONT_SIZE,
        "legend.fontsize": FONT_SIZE,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "axes.unicode_minus": False,
    }
)
IMAGE_COL = "Image"
METHOD_COL = "Method"
DESCRIPTOR_AGG = "first"
USE_IQR_FILTER = True
IQR_K = 1.5
MIN_REGRESSION_N = 10
TOP_SHAPE_N = 5
TARGET_N_TICKS = 6
METRIC_LABELS = {"FR": "FR", "OSR": "OR", "USR": "UR", "IoU": "IoU", "AP": "AP"}
SELECTED_METRICS = list(METRIC_LABELS.keys())
FIGURE1_FIXED_DESCRIPTOR_LABELS = {
    "mean_neighbor_count_within_2radius": "MNNR",
    "mean_radius1_inner_outer_intensity_ratio": "mLSBR",
    "mean_inner_intensity_cv": "mCV",
}
FIGURE1_FIXED_DESCRIPTORS = list(FIGURE1_FIXED_DESCRIPTOR_LABELS.keys())
MNND_DESCRIPTOR = "mean_nearest_neighbor_distance"
MNND_LABEL = "MNND"
ALL_FIXED_DESCRIPTORS = FIGURE1_FIXED_DESCRIPTORS + [MNND_DESCRIPTOR]
NON_SHAPE_DESCRIPTORS = {
    "mean_cell_volume",
    "mean_equivalent_radius",
    "mean_neighbor_count_within_2radius",
    "mean_nearest_neighbor_distance",
    "mean_foreground_intensity",
    "global_background_mean_intensity",
    "mean_global_contrast_ratio",
    "mean_local_background_intensity",
    "mean_local_contrast_ratio",
    "mean_radius1_inner_outer_intensity_ratio",
    "mean_inner_intensity_std",
    "mean_inner_intensity_cv",
    "mean_inner_intensity_entropy",
    "mean_inner_gradient_mean",
    "mean_inner_gradient_std",
    "mean_inner_intensity_morans_i",
    "median_inner_intensity_morans_i",
    "std_inner_intensity_morans_i",
    "num_labels_for_morans_i",
    "spacing_z_moran",
    "spacing_y_moran",
    "spacing_x_moran",
}
ALL_METRIC_COLS = ["FR", "IoU", "OSR", "USR", "MNR", "AP"]
META_COLS = [IMAGE_COL, METHOD_COL, "file", "batch", "source_path", "n_cells", "Tissue"]
FIGURE1_WIDTH = 16.0
FIGURE1_HEIGHT = 16.0
FIGURE2_WIDTH = 13.0
FIGURE2_HEIGHT = 16.0
POINT_SIZE = 10
POINT_ALPHA = 0.78
POINT_COLOR = "#808080"
REGRESSION_LINE_COLOR = "#D62728"
REGRESSION_LINE_WIDTH = 1.7
REGRESSION_CI = 95
WSPACE = 0.1
HSPACE = 0.5
FEATURE_LABEL_SIZE = 12
FEATURE_LABEL_PAD = 7
STATS_TEXT_SIZE = 12
STATS_TEXT_X = 0.0
STATS_TEXT_Y = 1.025
PDF_DPI = 300
PNG_DPI = 300
SHOW_FIGURES = True


In [ ]:
# Validate columns and convert numeric data.
if IMAGE_COL not in df.columns:
    raise ValueError(f"Image identifier column '{IMAGE_COL}' was not found.")
if METHOD_COL not in df.columns:
    raise ValueError(f"Method column '{METHOD_COL}' was not found.")
if DESCRIPTOR_AGG not in {"first", "mean", "median"}:
    raise ValueError("DESCRIPTOR_AGG must be 'first', 'mean', or 'median'.")
missing_metrics = [metric for metric in SELECTED_METRICS if metric not in df.columns]
if missing_metrics:
    raise ValueError("The following metrics were not found:\n" + "\n".join(missing_metrics))
candidate_descriptor_cols = [
    col
    for col in df.columns
    if col not in META_COLS + ALL_METRIC_COLS
    and (not col.startswith("std_"))
    and (not col.startswith("median_"))
]
all_descriptor_candidates = list(dict.fromkeys(ALL_FIXED_DESCRIPTORS + candidate_descriptor_cols))
numeric_columns = list(dict.fromkeys(SELECTED_METRICS + all_descriptor_candidates))
for col in numeric_columns:
    if col not in df.columns:
        continue
    df[col] = pd.to_numeric(df[col], errors="coerce")
descriptor_cols = [
    col for col in all_descriptor_candidates if col in df.columns and df[col].notna().any()
]
if len(descriptor_cols) == 0:
    raise ValueError("No valid numeric descriptor columns were found.")
missing_fixed_descriptors = [
    descriptor for descriptor in ALL_FIXED_DESCRIPTORS if descriptor not in descriptor_cols
]
if missing_fixed_descriptors:
    raise ValueError(
        "The following fixed descriptors were not found:\n" + "\n".join(missing_fixed_descriptors)
    )
print("Number of valid descriptor columns:", len(descriptor_cols))


In [ ]:
# Regression, sequential IQR filtering, and tick helpers.
def remove_iqr_outliers(data, cols, k=1.5):
    out = data.copy()
    # Filter the metric first, then the descriptor on the remaining samples.
    for col in cols:
        values = out[col].dropna()
        if values.empty:
            continue
        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1
        if pd.isna(iqr) or iqr == 0:
            continue
        lower = q1 - k * iqr
        upper = q3 + k * iqr
        out = out[(out[col] >= lower) & (out[col] <= upper)]
    return out


def format_p_value(p):
    if pd.isna(p):
        return "NA"
    if p < 0.001:
        return f"{p:.1e}"
    return f"{p:.3f}"


def calculate_linear_regression(plot_df, metric, descriptor):
    if (
        len(plot_df) < MIN_REGRESSION_N
        or plot_df[metric].nunique() < 2
        or plot_df[descriptor].nunique() < 2
    ):
        return None
    result = linregress(x=plot_df[descriptor].to_numpy(), y=plot_df[metric].to_numpy())
    return {
        "Slope": result.slope,
        "Intercept": result.intercept,
        "Slope_SE": result.stderr,
        "Slope_p": result.pvalue,
        "R_squared": result.rvalue**2,
    }


def prepare_plot_dataframe(source_df, metric, descriptor):
    if metric not in source_df.columns or descriptor not in source_df.columns:
        return (pd.DataFrame(columns=[metric, descriptor]), 0, 0)
    plot_df = source_df[[metric, descriptor]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    n_before_iqr = len(plot_df)
    if USE_IQR_FILTER:
        plot_df = remove_iqr_outliers(data=plot_df, cols=[metric, descriptor], k=IQR_K)
    n_after_iqr = len(plot_df)
    return (plot_df, n_before_iqr, n_after_iqr)


def detect_shape_descriptors(available_descriptors):
    return [
        descriptor
        for descriptor in available_descriptors
        if descriptor.startswith("mean_")
        and descriptor not in NON_SHAPE_DESCRIPTORS
        and (descriptor not in ALL_FIXED_DESCRIPTORS)
    ]


def make_shape_label(descriptor):
    label = str(descriptor)
    if label.startswith("mean_"):
        label = label[len("mean_") :]
    label = label.replace("_", " ")
    label = re.sub("(?<=[a-z0-9])(?=[A-Z])", " ", label)
    label = re.sub("\\s+", " ", label).strip()
    return label


def safe_filename(text):
    return re.sub("[^A-Za-z0-9_.-]+", "_", str(text))


def get_nice_step(raw_step):
    if not np.isfinite(raw_step) or raw_step <= 0:
        return 1.0
    exponent = np.floor(np.log10(raw_step))
    fraction = raw_step / 10**exponent
    if fraction <= 1:
        nice_fraction = 1
    elif fraction <= 2:
        nice_fraction = 2
    elif fraction <= 2.5:
        nice_fraction = 2.5
    elif fraction <= 5:
        nice_fraction = 5
    else:
        nice_fraction = 10
    return nice_fraction * 10**exponent


def calculate_axis_ticks(arrays, target_n_ticks=6):
    if len(arrays) == 0:
        return None
    valid_arrays = []
    for array in arrays:
        values = np.asarray(array, dtype=float)
        values = values[np.isfinite(values)]
        if len(values) > 0:
            valid_arrays.append(values)
    if len(valid_arrays) == 0:
        return None
    all_values = np.concatenate(valid_arrays)
    data_min = float(np.min(all_values))
    data_max = float(np.max(all_values))
    if data_min == data_max:
        raw_step = max(abs(data_min) * 0.1, 0.05)
    else:
        raw_step = (data_max - data_min) / max(target_n_ticks - 1, 1)
    step = get_nice_step(raw_step)
    tick_min = np.floor(data_min / step) * step
    tick_max = np.ceil(data_max / step) * step
    if tick_min > data_min or np.isclose(tick_min, data_min):
        tick_min -= step
    if tick_max < data_max or np.isclose(tick_max, data_max):
        tick_max += step
    ticks = np.arange(tick_min, tick_max + step * 0.5, step)
    ticks = np.round(ticks, 12)
    return ticks


In [ ]:
# Aggregate by image and rank shape descriptors separately for each metric.
aggregation_cols = list(dict.fromkeys([IMAGE_COL] + SELECTED_METRICS + descriptor_cols))
# Average metrics across selected methods; aggregate descriptors once per image.
aggregation_dict = {metric: "mean" for metric in SELECTED_METRICS}
for descriptor in descriptor_cols:
    aggregation_dict[descriptor] = DESCRIPTOR_AGG
image_level_df = (
    df[aggregation_cols]
    .replace([np.inf, -np.inf], np.nan)
    .groupby(IMAGE_COL, as_index=False)
    .agg(aggregation_dict)
)
print("Original dataframe rows:", len(df))
print("Image-level rows:", len(image_level_df))
method_count_per_image = df.groupby(IMAGE_COL)[METHOD_COL].nunique()
print("\nNumber of methods per image:")
print(method_count_per_image.value_counts().sort_index())
shape_descriptors = detect_shape_descriptors(descriptor_cols)
if len(shape_descriptors) < TOP_SHAPE_N:
    raise ValueError(
        f"Only {len(shape_descriptors)} valid shape descriptors were detected, but {TOP_SHAPE_N} are required."
    )
print("\nNumber of candidate shape descriptors:", len(shape_descriptors))
print("\nCandidate shape descriptors:")
for descriptor in shape_descriptors:
    print("  ", descriptor)
# Select each metric's top five by R-squared, then p-value and descriptor name.
all_metric_shape_ranking_dfs = []
metric_top5_shape_map = {}
for metric in SELECTED_METRICS:
    metric_ranking_rows = []
    for descriptor in shape_descriptors:
        current_df, n_before_iqr, n_after_iqr = prepare_plot_dataframe(
            source_df=image_level_df, metric=metric, descriptor=descriptor
        )
        regression_result = calculate_linear_regression(
            plot_df=current_df, metric=metric, descriptor=descriptor
        )
        if regression_result is None:
            continue
        metric_ranking_rows.append(
            {
                "Metric": metric,
                "Metric_label": METRIC_LABELS[metric],
                "Descriptor": descriptor,
                "Descriptor_label": make_shape_label(descriptor),
                "N_before_IQR": n_before_iqr,
                "N_after_IQR": n_after_iqr,
                **regression_result,
            }
        )
    metric_ranking_df = pd.DataFrame(metric_ranking_rows)
    if len(metric_ranking_df) < TOP_SHAPE_N:
        raise ValueError(
            f"Metric '{metric}' produced only {len(metric_ranking_df)} valid shape regressions, but {TOP_SHAPE_N} are required."
        )
    metric_ranking_df = metric_ranking_df.sort_values(
        ["R_squared", "Slope_p", "Descriptor"], ascending=[False, True, True]
    ).reset_index(drop=True)
    metric_ranking_df.insert(0, "Shape_rank", np.arange(1, len(metric_ranking_df) + 1))
    top5_descriptors = metric_ranking_df.head(TOP_SHAPE_N)["Descriptor"].tolist()
    metric_top5_shape_map[metric] = top5_descriptors
    all_metric_shape_ranking_dfs.append(metric_ranking_df)
    print("\n" + "=" * 80)
    print(f"Top {TOP_SHAPE_N} shape descriptors for {metric} ({METRIC_LABELS[metric]}):")
    print(
        metric_ranking_df[
            ["Shape_rank", "Descriptor", "Descriptor_label", "R_squared", "Slope_p", "N_after_IQR"]
        ]
        .head(TOP_SHAPE_N)
        .to_string(index=False)
    )
    metric_filename = safe_filename(metric)
    metric_ranking_file = os.path.join(
        save_path, f"{metric_filename}_shape_descriptor_R2_ranking.csv"
    )
    metric_ranking_df.to_csv(metric_ranking_file, index=False)
all_metric_shape_rankings_df = pd.concat(all_metric_shape_ranking_dfs, axis=0, ignore_index=True)
figure1_metric_descriptor_map = {}
figure2_metric_descriptor_map = {}
for metric in SELECTED_METRICS:
    current_top5 = metric_top5_shape_map[metric]
    figure1_metric_descriptor_map[metric] = FIGURE1_FIXED_DESCRIPTORS + current_top5[0:2]
    figure2_metric_descriptor_map[metric] = [MNND_DESCRIPTOR] + current_top5[2:5]
ALL_DESCRIPTOR_LABELS = {**FIGURE1_FIXED_DESCRIPTOR_LABELS, MNND_DESCRIPTOR: MNND_LABEL}
for metric in SELECTED_METRICS:
    for descriptor in metric_top5_shape_map[metric]:
        ALL_DESCRIPTOR_LABELS[descriptor] = make_shape_label(descriptor)
print("\n" + "=" * 80)
print("Metric-specific descriptors used in Figure 1 and Figure 2:")
for metric in SELECTED_METRICS:
    print(f"\n{metric} ({METRIC_LABELS[metric]})")
    print("  Figure 1:")
    for descriptor in figure1_metric_descriptor_map[metric]:
        print(f"    {descriptor} -> {ALL_DESCRIPTOR_LABELS[descriptor]}")
    print("  Figure 2:")
    for descriptor in figure2_metric_descriptor_map[metric]:
        print(f"    {descriptor} -> {ALL_DESCRIPTOR_LABELS[descriptor]}")


In [ ]:
# Prepare panel statistics and shared axis ranges.
def prepare_metric_specific_figure_data(image_level_source, metric_descriptor_map):
    plot_data = {}
    statistics_rows = []
    for metric in SELECTED_METRICS:
        row_descriptors = metric_descriptor_map[metric]
        for column_index, descriptor in enumerate(row_descriptors, start=1):
            current_df, n_before_iqr, n_after_iqr = prepare_plot_dataframe(
                source_df=image_level_source, metric=metric, descriptor=descriptor
            )
            regression_result = calculate_linear_regression(
                plot_df=current_df, metric=metric, descriptor=descriptor
            )
            plot_data[metric, descriptor] = current_df
            stats_row = {
                "Metric": metric,
                "Metric_label": METRIC_LABELS[metric],
                "Column": column_index,
                "Descriptor": descriptor,
                "Descriptor_label": ALL_DESCRIPTOR_LABELS[descriptor],
                "N_before_IQR": n_before_iqr,
                "N_after_IQR": n_after_iqr,
            }
            if regression_result is not None:
                stats_row.update(regression_result)
            else:
                stats_row.update(
                    {
                        "Slope": np.nan,
                        "Intercept": np.nan,
                        "Slope_SE": np.nan,
                        "Slope_p": np.nan,
                        "R_squared": np.nan,
                    }
                )
            statistics_rows.append(stats_row)
    statistics_df = pd.DataFrame(statistics_rows)
    all_used_descriptors = list(
        dict.fromkeys(
            (
                descriptor
                for metric in SELECTED_METRICS
                for descriptor in metric_descriptor_map[metric]
            )
        )
    )
    descriptor_xticks = {}
    for descriptor in all_used_descriptors:
        x_arrays = []
        for metric in SELECTED_METRICS:
            if descriptor not in metric_descriptor_map[metric]:
                continue
            current_df = plot_data[metric, descriptor]
            if current_df.empty:
                continue
            values = (
                current_df[descriptor]
                .replace([np.inf, -np.inf], np.nan)
                .dropna()
                .to_numpy(dtype=float)
            )
            if len(values) > 0:
                x_arrays.append(values)
        descriptor_xticks[descriptor] = calculate_axis_ticks(
            arrays=x_arrays, target_n_ticks=TARGET_N_TICKS
        )
    metric_yticks = {}
    for metric in SELECTED_METRICS:
        y_arrays = []
        for descriptor in metric_descriptor_map[metric]:
            current_df = plot_data[metric, descriptor]
            if current_df.empty:
                continue
            values = (
                current_df[metric].replace([np.inf, -np.inf], np.nan).dropna().to_numpy(dtype=float)
            )
            if len(values) > 0:
                y_arrays.append(values)
        metric_yticks[metric] = calculate_axis_ticks(arrays=y_arrays, target_n_ticks=TARGET_N_TICKS)
    return (plot_data, statistics_df, descriptor_xticks, metric_yticks)


In [ ]:
# Draw regression grids and export PDF, SVG, and PNG.
def draw_metric_specific_regression_figure(
    metric_descriptor_map,
    plot_data,
    descriptor_xticks,
    metric_yticks,
    figure_width,
    figure_height,
    output_prefix,
):
    nrows = len(SELECTED_METRICS)
    column_counts = [len(metric_descriptor_map[metric]) for metric in SELECTED_METRICS]
    if len(set(column_counts)) != 1:
        raise ValueError("Every metric row must contain the same number of descriptors.")
    ncols = column_counts[0]
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(figure_width, figure_height),
        sharex=False,
        sharey="row",
        squeeze=False,
    )
    for row_idx, metric in enumerate(SELECTED_METRICS):
        row_descriptors = metric_descriptor_map[metric]
        for col_idx, descriptor in enumerate(row_descriptors):
            ax = axes[row_idx, col_idx]
            current_df = plot_data[metric, descriptor]
            regression_result = calculate_linear_regression(
                plot_df=current_df, metric=metric, descriptor=descriptor
            )
            if regression_result is not None:
                sns.regplot(
                    data=current_df,
                    x=descriptor,
                    y=metric,
                    scatter_kws={
                        "s": POINT_SIZE,
                        "alpha": POINT_ALPHA,
                        "color": POINT_COLOR,
                        "edgecolors": "none",
                    },
                    line_kws={"color": REGRESSION_LINE_COLOR, "linewidth": REGRESSION_LINE_WIDTH},
                    ci=REGRESSION_CI,
                    ax=ax,
                )
                stats_text = f"R² = {regression_result['R_squared']:.2f}, p = {format_p_value(regression_result['Slope_p'])}"
            else:
                if not current_df.empty:
                    ax.scatter(
                        current_df[descriptor],
                        current_df[metric],
                        s=POINT_SIZE,
                        alpha=POINT_ALPHA,
                        color=POINT_COLOR,
                        edgecolors="none",
                    )
                stats_text = "R² = NA, p = NA"
            ax.text(
                STATS_TEXT_X,
                STATS_TEXT_Y,
                stats_text,
                transform=ax.transAxes,
                ha="left",
                va="bottom",
                fontsize=STATS_TEXT_SIZE,
                fontweight="normal",
                color="black",
                clip_on=False,
            )
            ax.set_xlabel(
                ALL_DESCRIPTOR_LABELS[descriptor],
                fontsize=FEATURE_LABEL_SIZE,
                fontweight="normal",
                labelpad=FEATURE_LABEL_PAD,
            )
            if col_idx == 0:
                ax.set_ylabel(
                    METRIC_LABELS[metric], fontsize=FONT_SIZE, fontweight="bold", labelpad=8
                )
            else:
                ax.set_ylabel("")
            x_ticks = descriptor_xticks.get(descriptor)
            if x_ticks is not None and len(x_ticks) >= 2:
                ax.set_xticks(x_ticks)
                ax.set_xlim(x_ticks[0], x_ticks[-1])
            y_ticks = metric_yticks.get(metric)
            if y_ticks is not None and len(y_ticks) >= 2:
                ax.set_yticks(y_ticks)
                ax.set_ylim(y_ticks[0], y_ticks[-1])
            ax.tick_params(
                axis="both",
                which="major",
                labelsize=FONT_SIZE,
                width=0.8,
                length=3.5,
                direction="out",
            )
            ax.tick_params(axis="x", labelbottom=True)
            ax.grid(False)
            sns.despine(ax=ax, top=True, right=True)
    for row_idx in range(nrows):
        for col_idx in range(1, ncols):
            axes[row_idx, col_idx].tick_params(labelleft=False)
    fig.subplots_adjust(
        left=0.075, right=0.995, top=0.975, bottom=0.055, wspace=WSPACE, hspace=HSPACE
    )
    fig.align_ylabels(axes[:, 0])
    pdf_file = os.path.join(save_path, f"{output_prefix}.pdf")
    svg_file = os.path.join(save_path, f"{output_prefix}.svg")
    png_file = os.path.join(save_path, f"{output_prefix}.png")
    fig.savefig(pdf_file, dpi=PDF_DPI, bbox_inches="tight")
    fig.savefig(svg_file, bbox_inches="tight")
    fig.savefig(png_file, dpi=PNG_DPI, bbox_inches="tight")
    print(f"\nSaved figure: {output_prefix}")
    print("PDF:", pdf_file)
    print("SVG:", svg_file)
    print("PNG:", png_file)
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)


In [ ]:
# Figure 1: three fixed descriptors and shape ranks 1-2.
figure1_plot_data, figure1_statistics_df, figure1_descriptor_xticks, figure1_metric_yticks = (
    prepare_metric_specific_figure_data(
        image_level_source=image_level_df, metric_descriptor_map=figure1_metric_descriptor_map
    )
)
draw_metric_specific_regression_figure(
    metric_descriptor_map=figure1_metric_descriptor_map,
    plot_data=figure1_plot_data,
    descriptor_xticks=figure1_descriptor_xticks,
    metric_yticks=figure1_metric_yticks,
    figure_width=FIGURE1_WIDTH,
    figure_height=FIGURE1_HEIGHT,
    output_prefix="figure1_metric_specific_MNNR_mLSBR_mCV_shape_rank1_to_rank2",
)


In [ ]:
# Figure 2: MNND and shape ranks 3-5.
figure2_plot_data, figure2_statistics_df, figure2_descriptor_xticks, figure2_metric_yticks = (
    prepare_metric_specific_figure_data(
        image_level_source=image_level_df, metric_descriptor_map=figure2_metric_descriptor_map
    )
)
draw_metric_specific_regression_figure(
    metric_descriptor_map=figure2_metric_descriptor_map,
    plot_data=figure2_plot_data,
    descriptor_xticks=figure2_descriptor_xticks,
    metric_yticks=figure2_metric_yticks,
    figure_width=FIGURE2_WIDTH,
    figure_height=FIGURE2_HEIGHT,
    output_prefix="figure2_metric_specific_MNND_shape_rank3_to_rank5",
)


In [ ]:
# Export feature selections, rankings, and regression statistics.
figure1_feature_rows = []
figure2_feature_rows = []
for metric in SELECTED_METRICS:
    current_top5 = metric_top5_shape_map[metric]
    for col_idx, descriptor in enumerate(figure1_metric_descriptor_map[metric], start=1):
        if descriptor in FIGURE1_FIXED_DESCRIPTORS:
            feature_type = "Fixed"
            shape_rank = np.nan
        else:
            feature_type = "Metric-specific shape descriptor"
            shape_rank = current_top5.index(descriptor) + 1
        figure1_feature_rows.append(
            {
                "Metric": metric,
                "Metric_label": METRIC_LABELS[metric],
                "Column": col_idx,
                "Descriptor": descriptor,
                "Descriptor_label": ALL_DESCRIPTOR_LABELS[descriptor],
                "Feature_type": feature_type,
                "Shape_rank_for_metric": shape_rank,
            }
        )
    for col_idx, descriptor in enumerate(figure2_metric_descriptor_map[metric], start=1):
        if descriptor == MNND_DESCRIPTOR:
            feature_type = "Fixed"
            shape_rank = np.nan
        else:
            feature_type = "Metric-specific shape descriptor"
            shape_rank = current_top5.index(descriptor) + 1
        figure2_feature_rows.append(
            {
                "Metric": metric,
                "Metric_label": METRIC_LABELS[metric],
                "Column": col_idx,
                "Descriptor": descriptor,
                "Descriptor_label": ALL_DESCRIPTOR_LABELS[descriptor],
                "Feature_type": feature_type,
                "Shape_rank_for_metric": shape_rank,
            }
        )
figure1_features_df = pd.DataFrame(figure1_feature_rows)
figure2_features_df = pd.DataFrame(figure2_feature_rows)
all_rankings_file = os.path.join(save_path, "all_metrics_metric_specific_shape_R2_rankings.csv")
figure1_statistics_file = os.path.join(
    save_path, "figure1_metric_specific_regression_statistics.csv"
)
figure2_statistics_file = os.path.join(
    save_path, "figure2_metric_specific_regression_statistics.csv"
)
figure1_features_file = os.path.join(save_path, "figure1_metric_specific_selected_features.csv")
figure2_features_file = os.path.join(save_path, "figure2_metric_specific_selected_features.csv")
all_metric_shape_rankings_df.to_csv(all_rankings_file, index=False)
figure1_statistics_df.to_csv(figure1_statistics_file, index=False)
figure2_statistics_df.to_csv(figure2_statistics_file, index=False)
figure1_features_df.to_csv(figure1_features_file, index=False)
figure2_features_df.to_csv(figure2_features_file, index=False)
print("\nSaved tables:")
print(all_rankings_file)
print(figure1_statistics_file)
print(figure2_statistics_file)
print(figure1_features_file)
print(figure2_features_file)
print("\nFinished.")
